# NOTE: This notebook uses local fallback paths for exploratory analysis.
# Production pipeline runs via run_pipeline.py or the Airflow DAG.



# Stage 5 — Visualize: 2SFCA Accessibility Maps

This notebook produces all final visualizations of the Gold-layer accessibility output.

**Outputs:**
1. Choropleth map with ICF facility overlay
2. Quartile accessibility map (identifying hot/cold spots)
3. Facility catchment circles (900m radius)

In [ ]:
import sys
sys.path.insert(0, "..")

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pipeline.config import load_config
from pipeline.ingest.census_api import ingest_census_blocks
from pipeline.ingest.cms_api import ingest_icf_facilities
from pipeline.validate.quality_gates import validate_population, validate_facilities
from pipeline.transform.sfca_2 import run_transform

config  = load_config("../config.yaml")
pop_gdf = validate_population(ingest_census_blocks(config), config)
fac_gdf = validate_facilities(ingest_icf_facilities(config), config)
result  = run_transform(pop_gdf, fac_gdf, config)

print(f"Result shape : {result.shape}")
print(f"Score range  : [{result['accessibility_norm'].min():.4f}, {result['accessibility_norm'].max():.4f}]")

## 1. Main Choropleth Map with ICF Overlay

In [ ]:
from pipeline.visualize import plot_accessibility_map
from IPython.display import Image

fig_path = plot_accessibility_map(result, facility_gdf=fac_gdf, config=config)
print(f"Saved to: {fig_path}")
Image(str(fig_path))

## 2. Quartile Map — Identifying Access Gaps

In [ ]:
result2 = result.copy()
nonzero_mask = result2["accessibility_norm"] > 0

result2["access_class"] = "No Access"
q1, q2, q3 = result2.loc[nonzero_mask, "accessibility_norm"].quantile([0.25, 0.5, 0.75])
result2.loc[nonzero_mask & (result2["accessibility_norm"] <= q1), "access_class"] = "Low (Q1)"
result2.loc[nonzero_mask & (result2["accessibility_norm"] > q1) & (result2["accessibility_norm"] <= q2), "access_class"] = "Medium-Low (Q2)"
result2.loc[nonzero_mask & (result2["accessibility_norm"] > q2) & (result2["accessibility_norm"] <= q3), "access_class"] = "Medium-High (Q3)"
result2.loc[nonzero_mask & (result2["accessibility_norm"] > q3), "access_class"] = "High (Q4)"

colors = {
    "No Access":        "#d0d0d0",
    "Low (Q1)":         "#4575b4",
    "Medium-Low (Q2)":  "#74add1",
    "Medium-High (Q3)": "#fdae61",
    "High (Q4)":        "#d73027",
}

fig, ax = plt.subplots(figsize=(9, 10))
for label, color in colors.items():
    subset = result2[result2["access_class"] == label]
    if len(subset):
        subset.plot(ax=ax, color=color, linewidth=0)

fac_gdf.plot(ax=ax, color="black", marker="^", markersize=30, zorder=5)

patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
patches.append(mpatches.Patch(color="black", label=f"ICF Facilities (n={len(fac_gdf)})"))
ax.legend(handles=patches, loc="lower left", fontsize=9, title="Access Class")
ax.set_title("ICF Accessibility Quartile Map — Washington DC", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("../outputs/figures/eda_05_quartile_map.png", dpi=150, bbox_inches="tight")
plt.show()

print(result2["access_class"].value_counts().to_string())

## 3. Facility Catchment Circles (900m radius)

In [ ]:
d0 = config["analysis"]["distance_threshold_m"]
catchments = fac_gdf.copy()
catchments["geometry"] = fac_gdf.geometry.buffer(d0)

fig, ax = plt.subplots(figsize=(9, 10))
result.plot(ax=ax, color="#f5f5f5", edgecolor="#cccccc", linewidth=0.2)
catchments.plot(ax=ax, color="steelblue", alpha=0.15, edgecolor="steelblue", linewidth=0.5)
fac_gdf.plot(ax=ax, color="red", marker="^", markersize=30, zorder=5)

patches = [
    mpatches.Patch(color="steelblue", alpha=0.4, label=f"{d0} m catchment"),
    mpatches.Patch(color="red", label=f"ICF Facilities (n={len(fac_gdf)})"),
]
ax.legend(handles=patches, loc="lower left", fontsize=9)
ax.set_title(f"ICF Facility Catchment Areas — {d0} m radius", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("../outputs/figures/eda_05_catchment_circles.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Final Summary Statistics

In [ ]:
scores = result["accessibility_norm"]
pop    = result["population"]
pop_weighted_mean = (scores * pop).sum() / pop.sum()

summary = {
    "Total census blocks"       : f"{len(result):,}",
    "Total population"          : f"{int(pop.sum()):,}",
    "Total ICF facilities"      : len(fac_gdf),
    "Total licensed beds"       : int(fac_gdf["CRTFD_BED_CNT"].sum()),
    "Catchment radius"          : f"{d0} m",
    "Zero-access blocks"        : f"{(scores==0).sum():,} ({(scores==0).mean()*100:.1f}%)",
    "Mean accessibility (raw)"  : f"{result['accessibility_score'].mean():.6f}",
    "Pop-weighted mean access"  : f"{pop_weighted_mean:.4f}",
    "Max accessibility score"   : f"{scores.max():.4f}",
}

for k, v in summary.items():
    print(f"{k:<35}: {v}")